# ChaoSSM — Phase 2c: Does Fine-Tuning Fragility Scale?

## Why this phase

Phase 2b's most striking finding: a short, aggressive fine-tuning burst (50 steps at 10x
learning rate) nearly erased chaossm_free's entire copy-task advantage over linear (accuracy
0.132 -> 0.056, back to linear's own level), and chaossm_corridor was, if anything, *less*
stable than chaossm_free (p=0.031). Everything in this project so far has been run at
state_dim=48 — a genuine open question is whether this fragility is a real property of the
chaotic mechanism, or a small-model artifact that would shrink or vanish at the scale the
eventual language-modeling study (§9.2) will actually use.

## Design

Identical protocol to Phase 2b's fine-tuning test — same task (copy), same burst (50 steps,
10x lr), same three arms, same statistical protocol — repeated at three state dimensions:
**48** (the pilot scale used everywhere else), **96**, and **192** (4x the pilot scale).
`d_model` is tied to `state_dim` at each size, preserving the same architecture proportions
used throughout this project rather than introducing a new free variable.

Two questions:
- **Does the copy-task advantage itself hold up at larger scale?** (a prerequisite — if chaos
  stops winning at larger N, the fragility question is moot for that regime)
- **Does the post-burst advantage retention improve with scale?** Defined as
  `(post-burst chaos-minus-linear gap) / (pre-burst chaos-minus-linear gap)` — 0 means the
  advantage is fully erased (Phase 2b's result at N=48), 1 means fully retained.

5 seeds per (arm, size) — reduced from Phase 2b's 6 to keep 3 sizes x 3 arms x 5 seeds = 45
runs within a single-GPU, low-cost budget, consistent with this phase's scope.

## Setup

In [1]:
import os, json, math, time, warnings, copy as pycopy
from dataclasses import dataclass, asdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import wilcoxon
from torch.func import jacrev

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE)

def setup_results_dir(project="ChaoSSM", phase="phase2c"):
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        base = f"/content/drive/My Drive/{project}/results/{phase}"
    except Exception as e:
        base = os.path.abspath(f"./{project}_results/{phase}")
        print(f"[info] Not in Colab ({type(e).__name__}); using local dir.")
    os.makedirs(base, exist_ok=True); print("results ->", base); return base

RESULTS = setup_results_dir()
def savefig(f,n): p=os.path.join(RESULTS,n); f.savefig(p,dpi=300,bbox_inches="tight"); print("saved",p)
def savetable(d,n): p=os.path.join(RESULTS,n); d.to_csv(p,index=False); print("saved",p)
def savejson(o,n):
    p=os.path.join(RESULTS,n)
    with open(p,"w") as fh: json.dump(o,fh,indent=2,default=str)
    print("saved",p)

torch 2.11.0+cu128 | device: cuda
Mounted at /content/drive
results -> /content/drive/My Drive/ChaoSSM/results/phase2c


## Configuration

In [2]:
SMOKE_TEST = False   # full run: 3 arms x 3 sizes x 5 seeds, copy task + finetune burst

@dataclass
class Config:
    state_dim: int = 48   # overwritten per-run by the size sweep below
    d_model: int = 48
    vocab: int = 16
    steps: int = 1200
    batch: int = 64
    lr: float = 3e-4
    bptt_k: int = 64
    grad_clip: float = 1.0
    gamma_init: float = 1.0
    warmup_frac: float = 0.10
    dual_lr: float = 0.05
    alpha_max: float = 50.0
    corridor_target: float = 0.15
    corridor_half_width: float = 0.10
    lyap_window: int = 4
    lyap_subsample: int = 3
    lyap_every: int = 20
    copy_len: int = 12
    arms: tuple = ("linear", "chaossm_free", "chaossm_corridor")
    sizes: tuple = (48, 96, 192)
    seeds: int = 5
    eval_batch: int = 256
    finetune_steps: int = 50
    finetune_lr_mult: float = 10.0

CFG = Config()
if SMOKE_TEST:
    CFG.steps = 150; CFG.seeds = 2; CFG.eval_batch = 64; CFG.finetune_steps = 10
    CFG.sizes = (48, 96)
    print("*** SMOKE TEST — pipeline check only ***")
print(json.dumps({k: str(v) for k, v in asdict(CFG).items()}, indent=2))

{
  "state_dim": "48",
  "d_model": "48",
  "vocab": "16",
  "steps": "1200",
  "batch": "64",
  "lr": "0.0003",
  "bptt_k": "64",
  "grad_clip": "1.0",
  "gamma_init": "1.0",
  "warmup_frac": "0.1",
  "dual_lr": "0.05",
  "alpha_max": "50.0",
  "corridor_target": "0.15",
  "corridor_half_width": "0.1",
  "lyap_window": "4",
  "lyap_subsample": "3",
  "lyap_every": "20",
  "copy_len": "12",
  "arms": "('linear', 'chaossm_free', 'chaossm_corridor')",
  "sizes": "(48, 96, 192)",
  "seeds": "5",
  "eval_batch": "256",
  "finetune_steps": "50",
  "finetune_lr_mult": "10.0"
}


## 1. Model — unchanged, parametrized by state_dim/d_model as always

In [3]:
def triangle_fold(x):
    return 1.0 - torch.abs(torch.remainder(x, 2.0) - 1.0)

class LinearMap(nn.Module):
    def __init__(self, N):
        super().__init__()
        W = torch.empty(N, N); nn.init.orthogonal_(W)
        self.W_lin = nn.Parameter(W * 0.5)
    def forward(self, h):
        return torch.tanh(h @ self.W_lin.T)

class ChaoticMap(nn.Module):
    def __init__(self, N, gamma_init=1.0):
        super().__init__()
        self.w_r = nn.Parameter(torch.zeros(N))
        W = torch.empty(N, N); nn.init.orthogonal_(W)
        self.W_mix = nn.Parameter(W * 0.1)
        inv = math.log(math.expm1(gamma_init))
        self.w_gamma = nn.Parameter(torch.tensor(float(inv)))
    def get_r(self): return 4.0 * torch.sigmoid(self.w_r)
    def get_gamma(self): return F.softplus(self.w_gamma)
    def forward(self, h):
        r = self.get_r()
        u = h @ self.W_mix.T + r * h * (1 - h)
        return triangle_fold(self.get_gamma() * u)

class ChaoSSMCell(nn.Module):
    def __init__(self, cfg, phi_kind):
        super().__init__()
        self.N = cfg.state_dim
        self.phi_kind = phi_kind
        self.phi = LinearMap(cfg.state_dim) if phi_kind == "linear" \
                   else ChaoticMap(cfg.state_dim, cfg.gamma_init)
        self.A = nn.Parameter(torch.diag(torch.empty(cfg.state_dim).uniform_(0.9, 1.1)))
        self.B = nn.Parameter(torch.randn(cfg.state_dim, cfg.d_model) / math.sqrt(cfg.d_model))
        self.W_g = nn.Parameter(torch.randn(cfg.state_dim, cfg.d_model) / math.sqrt(cfg.d_model))
        self.b_g = nn.Parameter(torch.zeros(cfg.state_dim))
    def step(self, h, x):
        g = torch.sigmoid(x @ self.W_g.T + self.b_g)
        raw = g * self.phi(h @ self.A.T) + (1 - g) * h + x @ self.B.T
        return triangle_fold(raw) if self.phi_kind != "linear" else torch.tanh(raw)
    def init_state(self, b, dev):
        dtype = self.A.dtype
        return torch.zeros(b, self.N, device=dev, dtype=dtype) if self.phi_kind == "linear" \
               else torch.full((b, self.N), 0.5, device=dev, dtype=dtype)

class ChaoSSMLM(nn.Module):
    def __init__(self, cfg, phi_kind, vocab_size, readout_size):
        super().__init__()
        self.cfg = cfg
        self.emb = nn.Embedding(vocab_size, cfg.d_model)
        self.cell = ChaoSSMCell(cfg, phi_kind)
        self.readout = nn.Linear(cfg.state_dim, readout_size)
    def forward(self, tok, return_traj=False, noise_std=0.0):
        B, T = tok.shape
        x = self.emb(tok)
        if noise_std > 0:
            x = x + torch.randn_like(x) * noise_std
        h = self.cell.init_state(B, tok.device)
        outs, traj = [], []
        for t in range(T):
            h = self.cell.step(h, x[:, t]); outs.append(self.readout(h))
            if return_traj: traj.append(h)
            if (t + 1) % self.cfg.bptt_k == 0: h = h.detach()
        lg = torch.stack(outs, 1)
        return (lg, torch.stack(traj, 1), x) if return_traj else lg

## 2. Estimator, task, and accuracy — unchanged from Phase 2b

In [4]:
def _full_jac_fn(model, x):
    return lambda z: model.cell.step(z.unsqueeze(0), x.unsqueeze(0))[0]

def lambda_window(model, traj, xseq, W, n_sample):
    B, T, N = traj.shape
    if T <= W + 1:
        return torch.zeros((), device=traj.device)
    lams = []
    for b in torch.randperm(B)[:n_sample]:
        t0 = int(torch.randint(0, T - W - 1, (1,)))
        M = torch.eye(N, device=traj.device, dtype=traj.dtype)
        for k in range(W):
            M = jacrev(_full_jac_fn(model, xseq[b, t0+k]))(traj[b, t0+k]) @ M
        lams.append(torch.log(torch.linalg.matrix_norm(M, ord=2) + 1e-12) / W)
    return torch.stack(lams).mean()

def make_copy_batch(cfg, batch, device, gen):
    L, V = cfg.copy_len, cfg.vocab
    sep, pad = V, V + 1
    src = torch.randint(0, V, (batch, L), device=device, generator=gen)
    inp = torch.cat([src, torch.full((batch,1), sep, device=device),
                     torch.full((batch,L), pad, device=device)], 1)
    tgt = torch.cat([torch.full((batch,L+1), -100, device=device), src], 1)
    return inp, tgt

VOCAB_SIZE = CFG.vocab + 2
CHANCE = 1.0 / CFG.vocab

def copy_accuracy(logits, tgt):
    mask = tgt != -100
    pred = logits.argmax(-1)
    return ((pred == tgt) & mask).float().sum().item() / mask.float().sum().clamp(min=1).item()

## 3. Trainer — same mechanics, `state_dim`/`d_model` now vary per run

In [5]:
def train_one(arm, size, seed, cfg):
    cfg = pycopy.deepcopy(cfg)
    cfg.state_dim = size; cfg.d_model = size
    torch.manual_seed(seed); np.random.seed(seed)
    gen = torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gen.manual_seed(seed)
    phi_kind = "linear" if arm == "linear" else "chaossm"
    model = ChaoSSMLM(cfg, phi_kind, VOCAB_SIZE, VOCAB_SIZE).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, betas=(0.9, 0.95))

    use_lyap = (arm == "chaossm_corridor")
    lo = hi = None
    if use_lyap:
        lo = cfg.corridor_target - cfg.corridor_half_width
        hi = cfg.corridor_target + cfg.corridor_half_width
    a_hi = a_lo = 0.0
    warmup_steps = int(cfg.warmup_frac * cfg.steps)
    diverged = False
    final_acc = float("nan"); lam_val = float("nan")

    for step in range(cfg.steps):
        inp, tgt = make_copy_batch(cfg, cfg.batch, DEVICE, gen)
        need_traj = (arm != "linear")
        out = model(inp, return_traj=need_traj)
        logits, traj, xseq = out if need_traj else (out, None, None)
        L_task = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), tgt.reshape(-1),
                                 ignore_index=-100)

        L_pen = torch.zeros((), device=DEVICE)
        if arm != "linear":
            in_warmup = step < warmup_steps
            lam = lambda_window(model, traj, xseq, cfg.lyap_window, cfg.lyap_subsample)
            lam_val = float(lam.detach())
            if use_lyap and not in_warmup:
                L_pen = a_hi * F.relu(lam - hi) + a_lo * F.relu(lo - lam)

        opt.zero_grad(set_to_none=True); (L_task + L_pen).backward()
        gn = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip).item()
        opt.step()

        if use_lyap and step >= warmup_steps:
            a_hi = float(np.clip(a_hi + cfg.dual_lr * (lam_val - hi), 0.0, cfg.alpha_max))
            a_lo = float(np.clip(a_lo + cfg.dual_lr * (lo - lam_val), 0.0, cfg.alpha_max))

        if not (math.isfinite(L_task.item()) and math.isfinite(gn)):
            diverged = True; break

        if step % 20 == 0:
            with torch.no_grad():
                final_acc = copy_accuracy(logits, tgt)

    return model, cfg, dict(arm=arm, state_dim=size, seed=seed, diverged=diverged,
                            final_acc=final_acc, lam_settled=lam_val)

## 4. Fine-tuning-stability test — identical protocol to Phase 2b

In [6]:
def eval_finetune_stability(model, cfg, arm, size, seed):
    model = pycopy.deepcopy(model)
    gen = torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gen.manual_seed(seed + 30_000)

    with torch.no_grad():
        inp0, tgt0 = make_copy_batch(cfg, cfg.eval_batch, DEVICE, gen)
        acc_before = copy_accuracy(model(inp0), tgt0)

    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr * cfg.finetune_lr_mult, betas=(0.9, 0.95))
    for _ in range(cfg.finetune_steps):
        inp, tgt = make_copy_batch(cfg, cfg.batch, DEVICE, gen)
        logits = model(inp)
        loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), tgt.reshape(-1), ignore_index=-100)
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        opt.step()

    with torch.no_grad():
        inp1, tgt1 = make_copy_batch(cfg, cfg.eval_batch, DEVICE, gen)
        acc_after = copy_accuracy(model(inp1), tgt1)

    return dict(arm=arm, state_dim=size, seed=seed, acc_before=acc_before, acc_after=acc_after,
               acc_drop=acc_before - acc_after)

## 5. Run — 3 arms × 3 sizes × 5 seeds, train then fine-tune-burst

In [ ]:
base_rows, ft_rows = [], []
total = len(CFG.arms) * len(CFG.sizes) * CFG.seeds
done = 0
for size in CFG.sizes:
    for arm in CFG.arms:
        for sd in range(CFG.seeds):
            model, run_cfg, s = train_one(arm, size, sd, CFG)
            base_rows.append(s)
            ft = eval_finetune_stability(model, run_cfg, arm, size, sd)
            ft_rows.append(ft)
            done += 1
            flag = " DIVERGED" if s["diverged"] else ""
            print(f"[{done}/{total}] size={size:4d} {arm:18s} seed={sd}  "
                  f"final_acc={s['final_acc']:.4f} (chance {CHANCE:.4f})  "
                  f"before={ft['acc_before']:.4f}  after={ft['acc_after']:.4f}  "
                  f"drop={ft['acc_drop']:+.4f}{flag}")

BASE = pd.DataFrame(base_rows); FT = pd.DataFrame(ft_rows)
savetable(BASE, "phase2c_base_summary.csv"); savetable(FT, "phase2c_finetune.csv")

[1/45] size=  48 linear             seed=0  final_acc=0.0742 (chance 0.0625)  before=0.0677  after=0.0563  drop=+0.0114
[2/45] size=  48 linear             seed=1  final_acc=0.0573 (chance 0.0625)  before=0.0615  after=0.0654  drop=-0.0039
[3/45] size=  48 linear             seed=2  final_acc=0.0885 (chance 0.0625)  before=0.0986  after=0.0807  drop=+0.0179
[4/45] size=  48 linear             seed=3  final_acc=0.0469 (chance 0.0625)  before=0.0560  after=0.0638  drop=-0.0078
[5/45] size=  48 linear             seed=4  final_acc=0.0521 (chance 0.0625)  before=0.0618  after=0.0605  drop=+0.0013
[6/45] size=  48 chaossm_free       seed=0  final_acc=0.1536 (chance 0.0625)  before=0.1647  after=0.0654  drop=+0.0993
[7/45] size=  48 chaossm_free       seed=1  final_acc=0.1523 (chance 0.0625)  before=0.1579  after=0.0648  drop=+0.0931
[8/45] size=  48 chaossm_free       seed=2  final_acc=0.1289 (chance 0.0625)  before=0.1530  after=0.0622  drop=+0.0908
[9/45] size=  48 chaossm_free       seed

## 6. Does the advantage persist, and does it hold up under fine-tuning, at each size?

**Advantage retention** = `(post-burst chaos-minus-linear accuracy gap) / (pre-burst gap)`.
0 = fully erased (Phase 2b's N=48 result); 1 = fully retained; negative = fine-tuning flips
the ordering entirely.

In [ ]:
def gap_and_retention(size, chaos_arm):
    lin = FT[(FT.state_dim==size)&(FT.arm=="linear")]
    cha = FT[(FT.state_dim==size)&(FT.arm==chaos_arm)]
    pre_gap = cha.acc_before.mean() - lin.acc_before.mean()
    post_gap = cha.acc_after.mean() - lin.acc_after.mean()
    retention = post_gap / pre_gap if abs(pre_gap) > 1e-6 else float("nan")
    return dict(state_dim=size, chaos_arm=chaos_arm, pre_gap=float(pre_gap), post_gap=float(post_gap),
               retention=float(retention))

RETAIN = pd.DataFrame([gap_and_retention(sz, arm) for sz in CFG.sizes
                       for arm in ("chaossm_free", "chaossm_corridor")])
print(RETAIN.round(4).to_string(index=False))
savetable(RETAIN, "phase2c_retention.csv")

In [9]:
def paired(size, a, b, col):
    x = FT[(FT.state_dim==size)&(FT.arm==a)].sort_values("seed")[col].values
    y = FT[(FT.state_dim==size)&(FT.arm==b)].sort_values("seed")[col].values
    m = np.isfinite(x) & np.isfinite(y); x, y = x[m], y[m]
    if len(x) < 3 or np.allclose(x, y):
        return dict(state_dim=size, pair=f"{a} vs {b}", metric=col, n=len(x), p=np.nan,
                    median_diff=float(np.median(x-y)) if len(x) else np.nan)
    st, p = wilcoxon(x, y)
    return dict(state_dim=size, pair=f"{a} vs {b}", metric=col, n=len(x), p=float(p),
               median_diff=float(np.median(x-y)))

rows = []
for size in CFG.sizes:
    rows.append(paired(size, "linear", "chaossm_free", "acc_drop"))
    rows.append(paired(size, "chaossm_corridor", "chaossm_free", "acc_drop"))
    rows.append(paired(size, "linear", "chaossm_free", "acc_before"))

ST = pd.DataFrame(rows)
print(ST.to_string(index=False))
savetable(ST, "phase2c_stats.csv")
print(f"\nmin achievable two-sided p at n={CFG.seeds}: {2/(2**CFG.seeds):.4f}")

 state_dim                             pair     metric  n      p  median_diff
        48           linear vs chaossm_free   acc_drop  5 0.0625    -0.087891
        48 chaossm_corridor vs chaossm_free   acc_drop  5 0.0625     0.014648
        48           linear vs chaossm_free acc_before  5 0.0625    -0.085612
        96           linear vs chaossm_free   acc_drop  5 0.8125     0.004557
        96 chaossm_corridor vs chaossm_free   acc_drop  5 0.0625     0.160482
        96           linear vs chaossm_free acc_before  5 0.3125     0.002279
       192           linear vs chaossm_free   acc_drop  5 0.3125     0.003581
       192 chaossm_corridor vs chaossm_free   acc_drop  5 0.8125     0.002279
       192           linear vs chaossm_free acc_before  5 0.0625     0.011068
saved /content/drive/My Drive/ChaoSSM/results/phase2c/phase2c_stats.csv

min achievable two-sided p at n=5: 0.0625


In [10]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.4))
colors = {"linear":"gray", "chaossm_free":"tab:blue", "chaossm_corridor":"tab:green"}

for arm in CFG.arms:
    means = [BASE[(BASE.state_dim==sz)&(BASE.arm==arm)].final_acc.mean() for sz in CFG.sizes]
    ax[0].plot(CFG.sizes, means, marker="o", label=arm, color=colors[arm])
ax[0].axhline(CHANCE, color="r", ls="--", label="chance")
ax[0].set(xlabel="state_dim", ylabel="accuracy (pre-finetune)", title="Does the copy-task advantage persist with scale?")
ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)

for arm in CFG.arms:
    means = [FT[(FT.state_dim==sz)&(FT.arm==arm)].acc_drop.mean() for sz in CFG.sizes]
    ax[1].plot(CFG.sizes, means, marker="o", label=arm, color=colors[arm])
ax[1].set(xlabel="state_dim", ylabel="fine-tune-induced accuracy drop", title="Does the fragility shrink with scale?")
ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)

for chaos_arm, ls in [("chaossm_free","-"), ("chaossm_corridor","--")]:
    d = RETAIN[RETAIN.chaos_arm==chaos_arm]
    ax[2].plot(d["state_dim"], d.retention, marker="o", ls=ls, label=chaos_arm)
ax[2].axhline(1.0, color="g", ls=":", label="fully retained")
ax[2].axhline(0.0, color="r", ls=":", label="fully erased")
ax[2].set(xlabel="state_dim", ylabel="advantage retention", title="Advantage retained after fine-tune burst")
ax[2].legend(fontsize=8); ax[2].grid(alpha=.3)
fig.tight_layout(); savefig(fig, "phase2c_scaling.png"); plt.close(fig)

saved /content/drive/My Drive/ChaoSSM/results/phase2c/phase2c_scaling.png


## 7. Verdict

| Observation | Reading |
|---|---|
| retention stays near 0 at all three sizes | The fragility is a real property of the chaotic mechanism, not a small-model artifact. Report it as a scale-independent limitation in §5.5/§9.3, and budget for it explicitly in the §9.2 scaling phase (e.g., lower fine-tuning learning rates for ChaoSSM checkpoints specifically). |
| retention increases meaningfully with size (e.g., toward 0.5+ by N=192) | The fragility is at least partly a small-model artifact. Note this as a positive update to §5.5/§9.3 — the scaling phase may not need special fine-tuning precautions, though this should still be re-verified once real LM-scale training is underway rather than assumed from a 4x-scale extrapolation. |
| the copy-task advantage itself shrinks or vanishes by N=192 | A different, more fundamental finding — chaos's win may itself be a small-model effect. This would need to be resolved before the fine-tuning question is even meaningful, and should be reported as the primary finding of this phase if it occurs. |
| chaossm_corridor remains less stable than chaossm_free at all sizes | Reinforces §5.5's finding that the corridor's accuracy benefit does not extend to fine-tuning robustness — not a pilot-scale artifact of the regularizer specifically. |

In [11]:
verdict = dict(
    smoke_test=SMOKE_TEST,
    base_accuracy_by_size=BASE.groupby(["state_dim","arm"]).final_acc.mean().round(4)
                              .reset_index().assign(key=lambda d: d["state_dim"].astype(str)+"|"+d["arm"])
                              .set_index("key")["final_acc"].to_dict(),
    finetune_drop_by_size=FT.groupby(["state_dim","arm"]).acc_drop.mean().round(4)
                            .reset_index().assign(key=lambda d: d["state_dim"].astype(str)+"|"+d["arm"])
                            .set_index("key")["acc_drop"].to_dict(),
    retention=RETAIN.to_dict("records"),
    stats=ST.to_dict("records"),
    config=dict(steps=CFG.steps, seeds=CFG.seeds, sizes=CFG.sizes,
                finetune_steps=CFG.finetune_steps, finetune_lr_mult=CFG.finetune_lr_mult),
)
savejson(verdict, "phase2c_verdict.json")
print(json.dumps(verdict, indent=2, default=str))

saved /content/drive/My Drive/ChaoSSM/results/phase2c/phase2c_verdict.json
{
  "smoke_test": false,
  "base_accuracy_by_size": {
    "48|chaossm_corridor": 0.1555,
    "48|chaossm_free": 0.1367,
    "48|linear": 0.0638,
    "96|chaossm_corridor": 0.1956,
    "96|chaossm_free": 0.0656,
    "96|linear": 0.0797,
    "192|chaossm_corridor": 0.0607,
    "192|chaossm_free": 0.0557,
    "192|linear": 0.0628
  },
  "finetune_drop_by_size": {
    "48|chaossm_corridor": 0.0979,
    "48|chaossm_free": 0.0794,
    "48|linear": 0.0038,
    "96|chaossm_corridor": 0.138,
    "96|chaossm_free": 0.0009,
    "96|linear": 0.0084,
    "192|chaossm_corridor": -0.0017,
    "192|chaossm_free": -0.0021,
    "192|linear": 0.0042
  },
  "retention": [
    {
      "state_dim": 48,
      "chaos_arm": "chaossm_free",
      "pre_gap": 0.07402343750000001,
      "post_gap": -0.0016276041666666574,
      "retention": -0.021987686895338483
    },
    {
      "state_dim": 48,
      "chaos_arm": "chaossm_corridor",
    